# Week 1 · LLM Basics and the API

**GDG AI Track · Cycle 1: Foundations and Evals**

Time: a 2-hour session, plus take-home work (Project 1, Component A).

## By the end of this notebook you can

- Run a notebook on your own laptop
- Save your work to GitHub
- Write small Python functions and read an error message
- Explain in plain words what an LLM and a token are
- Send a prompt to an AI model from Python, handle errors, and count tokens
- Say what **Project 1: Document Extractor** is, and what you build each week until Week 4

## Read this first

- **You do not need to understand every line of code.** When a cell says **Run**, just run it and look at the output.
- You only *write* code in cells marked **Your turn**, and each one is short.
- Run cells **from top to bottom** with `Shift + Enter`. Don't skip cells.
- If something is badly broken: restart the kernel and run all cells again (VS Code: **Restart**, then **Run All**).

| Label | Meaning |
|---|---|
| Read | Read and think |
| Run | Run the cell and look at the output |
| Your turn | Edit the code, then run the cell. It tells you PASS or FAIL |
| Warning | This is where people get stuck |

## Agenda (matches the session slides)

| Part | Topic | Minutes |
|---|---|---|
| 0 | Setup (done BEFORE class) | 0 |
| (slides) | Intro to the track | 10 |
| 3 | What is an LLM? | 30 |
| 1, 2 | Notebook, GitHub and just enough Python | 15 |
| 4 | Your first model call | 30 |
| 5 | Break it on purpose: errors and a hallucination hunt | 20 |
| 6, 7 | Project 1 brief and take-home | 15 |

New to Python? Do Part 2 before class. It is pre-work for anyone who needs the ramp.

# Part 0 · Set up your laptop (BEFORE class)

Everything runs on **your own laptop**. Setup takes 20 to 40 minutes the first time, mostly downloading. **Do it at home, on good internet.** If you finish Step 8 and everything says OK, you are ready.

## Step 1 · Install Python 3.11 or newer

Open a terminal and check what you have.

- **Windows:** press Start, type `PowerShell`, open it, run `py --version`
- **Mac:** press Cmd + Space, type `Terminal`, open it, run `python3 --version`
- **Linux:** open a terminal, run `python3 --version`

If it says 3.11 or higher, go to Step 2. If not, install Python from https://www.python.org/downloads/ and check again.

> Warning: **Windows:** on the first screen of the installer, tick **Add python.exe to PATH** before you click Install.

## Step 2 · Install Git

Download it from https://git-scm.com/downloads and accept the default options. Check with `git --version` in a new terminal.

## Step 3 · Install VS Code

Download it from https://code.visualstudio.com. Open it, click the Extensions icon on the left, and install two extensions by Microsoft: **Python** and **Jupyter**.

## Step 4 · Make your project folder and environment

An **environment** is a private set of installed tools for this project. In VS Code open **Terminal → New Terminal** and run these lines one at a time.

**Mac / Linux:**

```bash
mkdir gdg-ai-track
cd gdg-ai-track
python3 -m venv .venv
source .venv/bin/activate
```

**Windows (PowerShell):**

```powershell
mkdir gdg-ai-track
cd gdg-ai-track
py -m venv .venv
.venv\Scripts\Activate.ps1
```

You should now see `(.venv)` at the start of the line. If you don't, stop and ask for help. Every time you open a new terminal for this project, run the activate line again.

If PowerShell says "running scripts is disabled", run `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` once, then try again.

Put this notebook inside the `gdg-ai-track` folder. In VS Code use **File → Open Folder** and choose `gdg-ai-track`.

## Step 5 · Install the tools you need

With `(.venv)` showing, run:

```bash
python -m pip install --upgrade pip
python -m pip install google-genai python-dotenv ipykernel jupyterlab
```

This downloads a few hundred MB, so use good wifi.

## Step 6 · Create a kernel

The **kernel** is the Python process that runs the code in your notebook. Run this once:

```bash
python -m ipykernel install --user --name gdg-ai --display-name "GDG AI Track (Python 3)"
```

Then open this notebook and click **Select Kernel** (top right in VS Code) → **Jupyter Kernel** → **GDG AI Track (Python 3)**. If it isn't listed, press `Ctrl + Shift + P`, run **Developer: Reload Window**, and look again.

## Step 7 · Get your API key

An **API key** is your password for the AI model. Go to https://aistudio.google.com/apikey, sign in with Google, create a key and copy it. Then run the next cell. It asks for your key (hidden as you paste) and saves it safely in a file called `.env`. It also creates a `.gitignore` file, which stops your key from ever being uploaded to GitHub.

> Warning: Never paste your key into a code cell, a message or a GitHub file.

> Warning: **Privacy.** On Google's free tier, what you send may be used to improve Google's products (checked 4 Nov 2026). Never send passwords or private information in prompts.

In [ ]:
from pathlib import Path
import getpass

key = getpass.getpass("")
Path(".env").write_text("GEMINI_API_KEY=" + key.strip() + "\n")
Path(".gitignore").write_text(".env\n.venv/\n__pycache__/\n.ipynb_checkpoints/\n")
print("Done. Created .env and .gitignore in this folder.")

## Step 8 · Check your setup

Run the next cell. You want to see OK on every line.

In [1]:
import sys, os

print("Python:", sys.version.split()[0], "(you need 3.11 or newer)")

try:
    from dotenv import load_dotenv
    import google.genai
    load_dotenv()
    print("Tools: OK")
except ImportError:
    print("Tools: MISSING. You are probably on the wrong kernel (Step 6).")

if os.environ.get("GEMINI_API_KEY"):
    print("API key: OK")
else:
    print("API key: NOT FOUND. Run the key cell above, then restart the kernel.")

Python: 3.14.6 (you need 3.11 or newer)
Tools: OK
API key: OK


## If something goes wrong

| Problem | Fix |
|---|---|
| `python` or `py` is "not found" | Reinstall Python and tick **Add python.exe to PATH**, then open a **new** terminal |
| No `(.venv)` at the start of the line | Run the activate line from Step 4 again, inside `gdg-ai-track` |
| "Running scripts is disabled" (PowerShell) | `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned`, then open a new PowerShell |
| Notebook shows no kernel, or the wrong one | Redo Step 6, then pick the kernel again |
| `ModuleNotFoundError` in the notebook | Wrong kernel. Pick **GDG AI Track (Python 3)** |
| Install fails with network errors | Try another network, such as a phone hotspot |

**Internet:** Part 0 and Parts 4 to 7 need internet. Parts 1 to 3 work offline.

# Part 1 · Notebooks, Git and GitHub (15 min)

## Read · What is a notebook?

A **notebook** is a document made of **cells**:

- **Text cells** (like this one) hold explanations.
- **Code cells** hold Python. Click one and press `Shift + Enter` to run it. The output appears underneath.

The code runs on your laptop, using the **kernel** you chose in Part 0. The kernel remembers every variable you create.

> Warning: If you restart the kernel, it forgets everything and you must run the cells again from the top. If a cell fails with `NameError`, you probably skipped a cell or restarted the kernel.

## Run · Your first code cell

In [3]:
import sys
print("Python version:", sys.version.split()[0])
print("Hello from my own laptop!")

Python version: 3.14.6
Hello from my own laptop!


### Run · A tiny helper for the exercises

Run this once. It gives you a function called `check` that prints PASS or FAIL. You don't need to understand it yet.

In [2]:
RESULTS = {}

def check(label, passed):
    RESULTS[label] = bool(passed)
    print("PASS" if passed else "FAIL", "-", label)

def scoreboard():
    print("Passed", sum(RESULTS.values()), "of", len(RESULTS))

## Read · Git and GitHub

- A **repository** (repo) is a project folder whose history is saved.
- A **commit** is a saved snapshot of your files with a short message, like *"Add notes file"*.
- **Git** is the tool that saves commits on your laptop.
- **GitHub** is a website that stores your repo online so you can share it and never lose it.

When something breaks at 11pm, you can look at what changed since the last working commit. In this track you submit every project as a GitHub repo.

## Your turn · Set up GitHub (5 minutes)

1. Create a free account at github.com if you don't have one.
2. Create a **new repository** called `gdg-ai-track`. Tick **Add a README file**.
3. Ask your track lead whether it should be Public or Private.
4. Paste the repo's web address between the quotes below and run the cell.

> Warning: **Never put secrets in a repo.** Bots scan GitHub for leaked API keys within minutes. Your `.gitignore` from Part 0 protects you.

In [ ]:
MY_GITHUB_REPO = ""   # example: "https://github.com/yourname/gdg-ai-track"

check("GitHub repo address looks right", MY_GITHUB_REPO.startswith("https://github.com/") and len(MY_GITHUB_REPO) > 25)

### How to save this notebook to your repo

1. Save the notebook (`Ctrl + S`).
2. Open your repo on github.com and click **Add file → Upload files**.
3. Drag in the `.ipynb` file, write a message like "Add Week 1 notebook", and click **Commit changes**.

(Using `git push` from your laptop comes in a later week.)

## Read · Virtual environments

You made one in Part 0 (the `.venv` folder). It keeps this project's tools separate from other projects, so they never clash. Your kernel is the link between this notebook and that environment. That's all you need to know for now.

# Part 2 · The Python you need (30 min)

This is **not** a Python course. It is the small amount you need for Project 1, which turns messy documents into clean data. If you are new to Python, do this part before class.

Read, run, then do the short exercises. In each exercise cell, edit only the lines marked `TODO`, then run the cell again. The `check` lines at the bottom tell you PASS or FAIL.

## Read · Values and f-strings

Python stores information in **variables**. Common types: `str` (text), `int` (whole number), `float` (decimal), `bool` (`True` or `False`). An **f-string** (text starting with `f"`) lets you put variables inside text using `{curly braces}`.

In [ ]:
club = "GDG"
members = 42
is_open = True

print(f"{club} has {members} members.")
print(type(club), type(members), type(is_open))

## Read · Lists and dictionaries

- A **list** holds items in order: `["a", "b", "c"]`. Counting starts at 0, so `items[0]` is the first and `items[-1]` is the last.
- A **dictionary** (dict) holds `key: value` pairs: `{"title": "AI Workshop"}`. You look things up by key.

A campus notice is naturally a dict. You will see this shape a lot.

In [ ]:
topics = ["Python", "APIs", "Prompting"]
topics.append("Testing")
print(topics[0], "and", topics[-1])
print("How many topics:", len(topics))

notice = {
    "title": "AI Workshop",
    "date": "2026-11-12",
    "venue": "Hall B",
}
print(notice["title"])

notice["registration_required"] = True
print(notice)

## Read · Functions

A **function** is a named, reusable piece of code. It takes **inputs** (called parameters) and hands back a result with `return`.

- `print(...)` shows something on screen.
- `return ...` gives a value back so other code can use it.

In [ ]:
def shout(text):
    return text.upper() + "!"

print(shout("welcome to gdg"))

### Your turn · Exercise A: `word_count`

Make the function return how many words are in `text`.

Hint: `text.split()` turns text into a list of words, and `len(...)` counts the items in a list.

In [ ]:
def word_count(text):
    # TODO: replace the next line
    return None

# --- check (don't edit below) ---
check("word_count: simple sentence", word_count("GDG meets on Thursday") == 4)
check("word_count: extra spaces", word_count("  hello    world  ") == 2)
check("word_count: empty text", word_count("") == 0)

### Your turn · Exercise B: `make_notice`

Make the function return a dict with the keys `title`, `date`, `venue` and `registration_required`.

The last input has a **default value** (`False`), so callers can leave it out.

In [ ]:
def make_notice(title, date, venue, registration_required=False):
    # TODO: build and return a dict
    return None

# --- check (don't edit below) ---
expected = {"title": "Hack Night", "date": "2026-12-03", "venue": "Lab 2", "registration_required": True}
check("make_notice: all four keys", make_notice("Hack Night", "2026-12-03", "Lab 2", True) == expected)
check("make_notice: default is False", make_notice("Talk", "2026-11-19", "Hall A") == {"title": "Talk", "date": "2026-11-19", "venue": "Hall A", "registration_required": False})

## Read · Loops

A `for` loop repeats code for each item in a list. An `if` runs code only when something is true.

In [ ]:
notices = [
    {"title": "AI Workshop", "date": "2026-11-12", "venue": "Hall B", "registration_required": True},
    {"title": "Career Talk", "date": "2026-11-19", "venue": "Auditorium", "registration_required": False},
    {"title": "Hack Night", "date": "2026-12-03", "venue": "Lab 2", "registration_required": True},
]

for n in notices:
    print(n["date"], n["title"], "at", n["venue"])
    if n["registration_required"]:
        print("    (register first)")

## Read · JSON

**JSON** is a text format for data that programs send to each other. It looks almost like a Python dict.

- `json.dumps(x)` turns Python data into JSON **text** (dump**s** = to **s**tring).
- `json.loads(text)` turns JSON text back into Python data (load**s** = from **s**tring).

In [ ]:
import json

text = json.dumps(notices[0], indent=2)
print(text)

back = json.loads(text)
print("Back to a dict. Title:", back["title"])

### Your turn · Exercise C: round trip

Make `to_json_string` return pretty JSON text (use `indent=2`) and `from_json_string` turn JSON text back into Python.

In [ ]:
def to_json_string(obj):
    # TODO: use json.dumps with indent=2
    return None

def from_json_string(s):
    # TODO: use json.loads
    return None

# --- check (don't edit below) ---
check("to_json_string gives indented text", to_json_string({"a": 1}) == '{\n  "a": 1\n}')
check("from_json_string parses", from_json_string('{"a": [1, 2], "ok": true}') == {"a": [1, 2], "ok": True})
check("round trip keeps the data", from_json_string(to_json_string(notices)) == notices)

## Read · Files

You can save text to a file and read it back. Files appear in the same folder as this notebook, so look for `notices.json` in your file list after running the cell.

In [ ]:
from pathlib import Path

Path("notices.json").write_text(json.dumps(notices, indent=2))

loaded = json.loads(Path("notices.json").read_text())
print(len(loaded), "notices loaded. First title:", loaded[0]["title"])

### Your turn · Exercise D: save and load helpers

`save_json(data, path)` writes data to a file as pretty JSON. `load_json(path)` reads it back. You will use both in the take-home.

In [ ]:
def save_json(data, path):
    # TODO: write json.dumps(data, indent=2) to the file at path
    pass

def load_json(path):
    # TODO: read the file at path and return the data
    return None

# --- check (don't edit below) ---
save_json(notices, "exercise_d.json")
check("save_json and load_json work together", load_json("exercise_d.json") == notices)

## Read · Errors and how to read them

Your code **will** break. Professionals break code all day. The skill is staying calm and reading the message. When Python fails it prints a **traceback**. Here is an example:

```
Traceback (most recent call last):
  File "example.py", line 4, in <module>
    print(notice["venue"])
KeyError: 'venue'
```

**Read it from the bottom.** The last line says what went wrong: `KeyError: 'venue'` means "the dict has no key called venue". The lines above show where.

**When something breaks:**

1. Read the last line of the error.
2. Find the line of *your* code it points to.
3. Use `print()` to look at the variables on that line.
4. Change **one** thing and run again.
5. Still stuck? Search the last line online, or ask a teammate.

| Error | Usually means |
|---|---|
| `NameError` | A name doesn't exist (typo, or you didn't run an earlier cell) |
| `TypeError` | You mixed types, like text + number |
| `KeyError` | A dict doesn't have that key |
| `IndexError` | A list doesn't have that position |
| `ModuleNotFoundError` | A tool isn't installed, or you're on the wrong kernel |

## Your turn · Debug clinic

Each function below has one bug. Run the cell. Read what happens (an error, or FAIL). Find the bug and fix it.

### Bug 1

In [ ]:
def average_word_length(words):
    total = 0
    for w in words:
        total += len(w)
    return total / len(word)

# --- check (don't edit below) ---
check("average_word_length", average_word_length(["ab", "abcd"]) == 3.0)

### Bug 2

In [ ]:
def describe_attendance(count):
    return "Attendees: " + count

# --- check (don't edit below) ---
check("describe_attendance", describe_attendance(42) == "Attendees: 42")

### Bug 3

In [ ]:
def last_item(items):
    return items[len(items)]

# --- check (don't edit below) ---
check("last_item", last_item([1, 2, 3]) == 3)

### Bug 4 (the sneaky one)

This one gives **no error**, only wrong answers. That is why we test our code. Hint: `or` doesn't work the way it sounds in English. Try `day == "Saturday" or day == "Sunday"`.

In [ ]:
def is_weekend(day):
    return day == "Saturday" or "Sunday"

# --- check (don't edit below) ---
check("is_weekend: Sunday", is_weekend("Sunday") == True)
check("is_weekend: Monday", is_weekend("Monday") == False)

## Check · Part 2 scoreboard

Run this to see how you did. If something shows FAIL, go back and fix it before moving on.

In [ ]:
scoreboard()

# Part 3 · What is an LLM? (15 min)

## Read · The big idea: predict the next piece of text

A **large language model (LLM)** is trained on a huge amount of text to do one thing: given the text so far, predict what comes next. It picks one next piece, adds it, and repeats. Do that thousands of times and you get a whole answer.

```
"The capital of Nigeria is"        → model picks " Abuja"
"The capital of Nigeria is Abuja"  → model picks the next piece again ...
```

Chat assistants are LLMs that were also trained to follow instructions. For now, remember this: **an LLM generates plausible text. It does not look things up and it does not check facts.**

## Run · A toy language model

Here is the same idea in miniature. This toy has no learning. It only knows which word can follow which word, and it picks one at random. Run the cell **several times**. Then change the list under `"<start>"` to just `["please"]` and run it again.

In [ ]:
import random

next_words = {
    "<start>": ["the", "all", "please"],
    "the": ["workshop", "deadline"],
    "all": ["students", "members"],
    "please": ["register", "bring"],
    "workshop": ["starts"], "deadline": ["is"],
    "students": ["must"], "members": ["must"],
    "starts": ["on"], "is": ["on"],
    "on": ["thursday", "friday"],
    "must": ["register", "attend"],
    "register": ["today"], "attend": ["today"], "bring": ["laptops"],
    "thursday": ["<end>"], "friday": ["<end>"], "today": ["<end>"], "laptops": ["<end>"],
}

word = "<start>"
sentence = []
while True:
    word = random.choice(next_words[word])
    if word == "<end>":
        break
    sentence.append(word)

print(" ".join(sentence))

Every sentence sounds believable, but nothing checks whether a workshop is really happening on that day. Keep that in mind when we reach hallucination.

## Read · Tokens

Models don't read letters or words. They read **tokens**: small pieces of text, often a word or part of a word. In English, one token is roughly ¾ of a word.

Tokens matter because **you pay per token** (or use up a free quota), and the model can only handle a limited number of them at once. In Part 4 you will see the real token count for your own prompts.

Rule of thumb: about 100 tokens for 75 English words.

**Think about it:** Yoruba and Pidgin often need *more* tokens than English for the same meaning, so the same message can cost more. Is that fair? You will measure it yourself in the take-home.

## Read · Context window

The **context window** is the most text (in tokens) the model can handle in one request. Your instructions, any text you paste in, and the answer must all fit.

> Warning: **The model has no memory between calls.** Each call is separate. Chat apps *seem* to remember because they send the whole conversation again every time. You'll prove this in Part 4.

## Read · Temperature

**Temperature** controls how adventurous the model is. Low: predictable answers. High: more varied (and sometimes odder) answers.

## Read · Hallucination

Because the model generates *plausible* text rather than *checked* facts, it can say false things confidently. This is called **hallucination**. It is a normal property of LLMs, not a rare glitch. This is why this track keeps coming back to **testing**.

## Check · Quick questions (think first, then click)

**1. What does an LLM do at each step?**

<details><summary>Answer</summary>It predicts the next piece of text, adds it, and repeats.</details>

**2. True or false: if I tell the API my name in one call, it remembers it in the next call.**

<details><summary>Answer</summary>False. Each call is separate unless you send the earlier messages again.</details>

**3. Why can an LLM give a confident but wrong answer?**

<details><summary>Answer</summary>It generates plausible text from patterns. It doesn't look facts up or verify them.</details>

# Part 4 · Your first model call (30 min)

An **API** lets your code talk to a service. You made your API key in Part 0. We use Google's Gemini model, which has a free tier.

> Warning: **Privacy.** Never send passwords, personal data or private student information in prompts. Campus notices and made-up examples are fine.

> Warning: **Free tiers have limits.** If you see a `429` error, you are going too fast or used your quota. Wait a minute and try again.

## Run · Connect to the model

This cell reads your key from `.env` and connects. It needs internet.

In [ ]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()   # finds GEMINI_API_KEY by itself

MODEL = "gemini-3.8-flash"
print("Ready. Model:", MODEL)

> Warning: **Model names change.** If you get a `404` or "model not found" error, run the next cell, pick a name from the list, and put it in `MODEL` above.

### Run · Which models can I use? (only if you got a 404)

In [ ]:
for m in client.models.list():
    if "flash" in m.name:
        print(m.name.replace("models/", ""))

## Run · Your first call

This is the moment. Run it. Three parts: **client** is who you are, **model** is which AI, **contents** is what you ask.

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents="Explain what a large language model is in two sentences, for a first-year student.",
)
print(response.text)

## Read · What comes back?

The reply is more than text. It also tells you what the call used. Remember: tokens are what limits and costs are counted in. Longer prompts and longer answers use more tokens.

In [ ]:
usage = response.usage_metadata
print("Tokens you sent:", usage.prompt_token_count)
print("Tokens that came back:", usage.candidates_token_count)

## Run · A shortcut: the `ask` function

Calling the model the long way each time is tiring, so here is a shortcut. `ask("your prompt")` sends the prompt and gives you back the answer and the token usage. You can ignore how it works for now. Just run it and use it.

In [ ]:
from google.genai import types

def ask(prompt, system=None, temperature=None):
    config = types.GenerateContentConfig(system_instruction=system, temperature=temperature)
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    return response.text or "", response.usage_metadata

text, usage = ask("Reply with exactly one word: ready")
print(text.strip())

## Run · Change one knob at a time

You can change three things: the **prompt**, the **system instruction** (sets the role and tone), and the **temperature**. Change **one at a time**, or you won't know which one mattered.

Here is the same prompt, plain and then with a system instruction:

In [ ]:
prompt = "Explain a token to a 10-year-old."

text, _ = ask(prompt)
print("PLAIN:\n", text.strip(), "\n")

text, _ = ask(prompt, system="You are a patient tutor.")
print("WITH A SYSTEM INSTRUCTION:\n", text.strip())

### Run · Knob 3: temperature

The same prompt three times at temperature 0, then three times at temperature 1. Guess what you'll see before you run it. Low temperature suits extraction and code. High temperature suits names and slogans.

In [ ]:
import time

prompt = "Suggest one name for a campus AI club. Reply with the name only."

for temp in (0.0, 1.0):
    print("--- temperature =", temp)
    for i in range(3):
        text, _ = ask(prompt, temperature=temp)
        print("  ", text.strip())
        time.sleep(1)

## Run · The model has no memory

Three calls. Look closely at call 2 and call 3.

In [ ]:
text1, _ = ask("My name is Tunde. Please remember it.")
print("Call 1:", text1.strip(), "\n")

text2, _ = ask("What is my name?")
print("Call 2 (a new, separate call):", text2.strip(), "\n")

history = "Earlier in our chat, the user said: My name is Tunde.\n\nUser: What is my name?"
text3, _ = ask(history)
print("Call 3 (we sent the history again):", text3.strip())

Call 2 has no idea who you are. Call 3 "remembers" only because *we pasted the history back in*. Every chatbot works this way.

---

# Part 5 · Break it on purpose (20 min)

Real apps call models all day, and calls fail. Errors are normal. Your job is to know what to do.

| What you see | What it means | What to do |
|---|---|---|
| Bad or missing key | The key is wrong, or not found | Check the `.env` file name and the key |
| `404` | The model name doesn't exist | Fix `MODEL` |
| `429` | Too many requests (rate limit) | Wait, then try again |
| `500` or `503` | The server is busy. Not your fault | Retry after a short, growing pause |

**Read the error message first.** It usually tells you what to fix.

## Run · Cause two errors safely

`try` / `except` means: *try this code, and if it fails, don't crash. Run the `except` part instead.* This is how real programs survive errors. You will use it in the take-home.

In [ ]:
# 1. A model name that doesn't exist
try:
    client.models.generate_content(model="not-a-real-model", contents="hi")
except Exception as e:
    print("Wrong model name ->", type(e).__name__)
    print(str(e)[:200], "\n")

# 2. A key that doesn't work
bad_client = genai.Client(api_key="this-is-not-a-real-key")
try:
    bad_client.models.generate_content(model=MODEL, contents="hi")
except Exception as e:
    print("Bad key ->", type(e).__name__)
    print(str(e)[:200])

Look at the two messages. Which one told you more clearly what to fix?

## Run · Hallucination hunt

The event below **does not exist** (it is set in 2031). Ask about it two ways and compare. Remember: a fluent sentence and a true sentence look the same to the model.

In [ ]:
plain, _ = ask("Summarise the main findings of the final report of the 2031 Lagos Inter-University AI Hackathon.")
print("PLAIN QUESTION:\n", plain.strip(), "\n")
time.sleep(1)

pushy, _ = ask("List exactly 3 key findings from the final report of the 2031 Lagos Inter-University AI Hackathon. Do not say you don't know.")
print("PUSHY QUESTION:\n", pushy.strip())

### Your turn · Reflect (write 2 or 3 sentences)

Did either answer invent details? What does this tell you about trusting an AI tool's output *without testing it*? What can you do about it? (Ideas: give it the facts to use, ask for sources and check them, test on cases where you know the answer.)

In [ ]:
my_reflection = """

"""

# Part 6 · Project 1 brief: Document Extractor (5 min)

**Project 1 is due in Week 4. You work solo.** The brief:

> Take a real stream of messy documents (course outlines, event notices, CVs, receipts) and build a tool that turns each one into **validated JSON**.

Here is a typical messy document (made up):

In [ ]:
messy_notice = """HEY GUYS!!! GDG AI workshop this Thursday 12th Nov @ Hall B, 4pm sharp. Bring laptops.
Register at the link in bio. No registration = no entry! pls share"""
print(messy_notice)

Your tool should turn it into something like this, **every time**, for any document of that kind:

```json
{
  "title": "GDG AI workshop",
  "date": "12 Nov",
  "time": "16:00",
  "venue": "Hall B",
  "registration_required": true
}
```

Here is what a simple attempt gives us:

In [ ]:
text, _ = ask("Extract the title, date, time and venue from this notice:\n\n" + messy_notice)
print(text)

Could a program read that reliably? The notice doesn't say the year, and someone has to decide that "4pm" means `16:00`. That is the problem you will solve.

## How the next four weeks build Project 1

| Week | Topic | What you build (take-home) |
|---|---|---|
| 1 (today) | LLM basics and the API | **Component A:** an API client script with error handling, plus a token explorer |
| 2 | Prompting and structured output | **Component B:** an extractor with schema validation, tested on 20 messy inputs |
| 3 | Evals 101 | **Component C:** an eval harness for the extractor, with one measured prompt improvement |
| 4 | Project session | Put A, B and C together. Demos, peer code review, live fix time |

## What you hand in at Week 4

- A repo with a README and one-command setup
- A **30-case golden set** (30 documents where you know the right answer)
- Your eval harness, and an **eval report with 2 measured improvements**
- A **1-page "what broke" write-up**

## How Project 1 is graded

| What | Weight |
|---|---|
| Working product (runs from the README on a clean machine) | 25% |
| Evaluation rigour (golden set, numbers before and after) | 25% |
| Carry-forward use (reuses earlier work correctly) | 15% |
| Failure analysis and write-up | 15% |
| Oral defence and individual contribution | 20% |

> The eval harness you build in Week 3 gets reused in every later project. What you build now matters for the whole track.

**Start now:** begin a folder of messy documents (event notices, course outlines, receipts). Use made-up or public ones. Never use anything private.

---

# Part 7 · Take-home: Project 1, Component A

**Component A = an API client script with error handling, plus a token explorer.**

You will build three small things, then save and submit:

1. `safe_ask`: a function that calls the model and **never crashes**
2. Your own 5 prompts, run through `safe_ask` and saved to `results.json`
3. A token explorer that counts tokens for different kinds of text

## Step 1 · Build `safe_ask`

This function calls the model. If the call fails, it waits and tries again (up to 3 attempts). If it still fails, it returns an error instead of crashing. It always returns a dict.

On success, the dict has keys `prompt`, `response`, `model`, `prompt_tokens`, `output_tokens`. On failure, it has `prompt` and `error`.

(Real apps retry only on `429`, `500` and `503`. We retry everything to keep it simple.)

<details><summary>Stuck? Click for the full answer</summary>

```python
def safe_ask(prompt):
    for attempt in range(3):
        try:
            text, usage = ask(prompt)
            return {
                "prompt": prompt,
                "response": text,
                "model": MODEL,
                "prompt_tokens": usage.prompt_token_count,
                "output_tokens": usage.candidates_token_count,
            }
        except Exception as e:
            if attempt == 2:
                return {"prompt": prompt, "error": str(e)}
            time.sleep(2 ** (attempt + 1))
```

</details>

In [ ]:
def safe_ask(prompt):
    for attempt in range(3):
        try:
            # TODO 1: call ask(prompt) to get text and usage
            # TODO 2: return a dict with keys: prompt, response, model, prompt_tokens, output_tokens
            pass
        except Exception as e:
            # TODO 3: if this was the last attempt (attempt == 2), return {"prompt": prompt, "error": str(e)}
            #         otherwise wait with time.sleep(2 ** (attempt + 1)) and let the loop try again
            pass

# --- check (don't edit below) ---
good = safe_ask("Reply with one word: ready")
check("safe_ask returns an answer", isinstance(good, dict) and "response" in good)

real_model = MODEL
MODEL = "not-a-real-model"       # break it on purpose
bad = safe_ask("hi")             # takes about 6 seconds: it retries
MODEL = real_model
check("safe_ask survives a failing call", isinstance(bad, dict) and "error" in bad)

## Step 2 · Run your own 5 prompts

Write 5 prompts of your own: one to **summarise** a text, one to **classify** something (for example, label a notice as *event*, *deadline* or *announcement*), one to **extract** facts from text, one to **rewrite** text in a different tone, and one **question whose answer you can check**. Replace each `TODO`.

In [ ]:
prompts = [
    "TODO: prompt 1 (summarise)",
    "TODO: prompt 2 (classify)",
    "TODO: prompt 3 (extract)",
    "TODO: prompt 4 (rewrite)",
    "TODO: prompt 5 (a question you can check)",
]

Now run them all through `safe_ask` and save the results. (`save_json` is the helper you wrote in Exercise D.)

In [ ]:
results = []

for prompt in prompts:
    # TODO 1: call safe_ask(prompt) and add what it gives back to results
    time.sleep(1)

# TODO 2: save results to "results.json" with save_json

## Step 3 · The token explorer

Your token explorer counts tokens for five kinds of text, so you can see which ones cost more. Fill in `count_tokens`, run the cell, then answer the questions below.

Hint: `client.models.count_tokens(model=MODEL, contents=text).total_tokens`

In [ ]:
samples = {
    "English": "Please send the minutes of the meeting before Friday.",
    "Pidgin":  "Abeg send the minutes of the meeting before Friday.",
    "Yoruba":  "Ẹ jọ̀ọ́ ẹ fi ìwé ìpàdé ránṣẹ́ kí ó tó di ọjọ́ Ẹtì.",
    "Code":    "def add(a, b):\n    return a + b",
    "Numbers": "2026-11-12 16:00 +234 801 234 5678",
}

def count_tokens(text):
    # TODO: return the number of tokens in text
    return None

for name, text in samples.items():
    print(name, "| characters:", len(text), "| tokens:", count_tokens(text))

Write your answers (a few sentences each):

1. Which text uses the most tokens for its length? Why might that be?
2. English, Pidgin and Yoruba mean about the same thing here. What does the token count mean for someone writing in Yoruba if an app charges per token? Is that fair?

In [ ]:
TOKEN_NOTES = """
1.
2.
"""

## Step 4 · Check your work

In [ ]:
saved = []
file_text = ""
if Path("results.json").exists():
    file_text = Path("results.json").read_text()
    saved = json.loads(file_text)

required = {"prompt", "response", "model", "prompt_tokens", "output_tokens"}
complete = len(saved) > 0
for r in saved:
    if not (required <= set(r) or "error" in r):
        complete = False

wrote_own = True
for p in prompts:
    if "TODO" in p:
        wrote_own = False

key = os.environ.get("GEMINI_API_KEY", "")

check("results.json exists", Path("results.json").exists())
check("it has 5 entries", len(saved) == 5)
check("every entry is complete (or recorded an error)", complete)
check("you wrote your own prompts", wrote_own)
check("no API key inside results.json", key == "" or key not in file_text)
check("token explorer counts tokens", isinstance(count_tokens("hello"), int))
check("you answered the token questions", len(TOKEN_NOTES.strip()) >= 40)

Open `results.json` in VS Code and read it. Do the answers make sense? Real engineers look at their data.

## Step 5 · What broke?

Write **one paragraph** about something that went wrong or surprised you this week (an error, a strange answer, a confusing step) and how you dealt with it. If nothing broke, describe the most surprising answer the model gave. This is practice for the 1-page "what broke" write-up you hand in at Week 4.

In [ ]:
WHAT_BROKE = """

"""

check("you wrote your 'what broke' paragraph", len(WHAT_BROKE.strip()) >= 80)
scoreboard()

## Step 6 · Submit

1. Open your `gdg-ai-track` repo on github.com → **Add file → Upload files**.
2. Upload `results.json` and this notebook. Write a commit message and click **Commit changes**.
3. Send your track lead the repo link, plus your "what broke" paragraph, before the next session.

> Warning: Before uploading the notebook, scroll through its outputs. If your API key ever appeared on screen, delete that output and make a new key.

---

# Glossary

| Term | Plain meaning |
|---|---|
| Notebook | A document of text cells and code cells |
| Kernel | The Python process that runs your code and remembers your variables |
| Environment (`.venv`) | A private folder of tools for one project |
| Repository (repo) | A project folder whose history Git saves |
| Commit | A saved snapshot of your files, with a message |
| Function | A named, reusable piece of code |
| Dict | Key and value pairs: `{"title": "..."}` |
| JSON | A text format for data that programs share |
| Traceback | Python's error report. Read it from the bottom |
| try / except | Run code, and if it fails, handle the error instead of crashing |
| LLM | A model trained to predict the next piece of text |
| Token | A small piece of text the model reads and writes |
| Context window | The most text the model can handle in one request |
| Temperature | A setting for how varied the answers are |
| Hallucination | A confident, believable, false statement |
| API / API key | A way for code to use a service / your password for it |
| Golden set | Documents where you know the right answer, used to test your tool |

# Next week

**Week 2: Prompting and structured output.** You'll turn messy text into validated JSON and start **Component B**: an extractor with schema validation, tested on 20 messy inputs.